# SVC Representation Search — Google Colab

Standalone research notebook. The previous SVC notebook remains untouched. Use **Runtime → Run all**.

This notebook searches SVC representations, PCA and RBF hyperparameters. `test.csv` is never read.

In [ ]:
!pip -q install scikit-learn pandas numpy matplotlib
from pathlib import Path
import time, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import f1_score
SEED = 42
print('Colab environment ready — SVC representation search')

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print('Drive mount unavailable:', e)
candidates = [Path('/content/data'), Path('/content/drive/MyDrive/hackathon/data'), Path('/content/drive/MyDrive/data')]
DATA_DIR = next((p for p in candidates if (p / 'train.csv').exists() and (p / 'val.csv').exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError('Put train.csv and val.csv in MyDrive/hackathon/data/ or /content/data/')
train = pd.read_csv(DATA_DIR / 'train.csv')
val = pd.read_csv(DATA_DIR / 'val.csv')
assert 'label' not in train.columns
assert {'label', 'engine_id'}.issubset(val.columns)
assert set(train.engine_id).isdisjoint(set(val.engine_id))
freq = sorted([c for c in train.columns if c.startswith('mV_')], key=lambda x: int(x.split('_')[1]))
print('DATA_DIR:', DATA_DIR)
print('train:', train.shape, 'val:', val.shape, 'spectrum:', len(freq))

In [ ]:
def spectrum(df):
    return df[freq].copy().interpolate(axis=1, limit_direction='both').fillna(0).to_numpy(dtype=float)

def build_rep(a, kind):
    if kind == 'raw':
        return a
    if kind == 'raw_d1':
        return np.column_stack([a, np.diff(a, axis=1)])
    if kind == 'raw_d1_d2':
        return np.column_stack([a, np.diff(a, axis=1), np.diff(a, n=2, axis=1)])
    if kind == 'centered':
        return a - a.mean(axis=1, keepdims=True)
    if kind == 'z_spectrum':
        mu = a.mean(axis=1, keepdims=True)
        sd = a.std(axis=1, keepdims=True)
        return (a - mu) / np.where(sd < 1e-8, 1.0, sd)
    if kind == 'd1':
        return np.diff(a, axis=1)
    if kind == 'd2':
        return np.diff(a, n=2, axis=1)
    if kind == 'shape_d1':
        mu = a.mean(axis=1, keepdims=True)
        sd = a.std(axis=1, keepdims=True)
        z = (a - mu) / np.where(sd < 1e-8, 1.0, sd)
        return np.column_stack([z, np.diff(z, axis=1)])
    raise ValueError(kind)

A = spectrum(train)
V = spectrum(val)
REP_KINDS = ['raw', 'raw_d1', 'raw_d1_d2', 'centered', 'z_spectrum', 'd1', 'd2', 'shape_d1']
print({k: build_rep(V, k).shape[1] for k in REP_KINDS})

In [ ]:
gkf = GroupKFold(n_splits=min(5, val.engine_id.nunique()))

def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro', zero_division=0)

def make_svc(C, gamma, pca=None):
    steps = [('scale', StandardScaler())]
    if pca is not None:
        steps.append(('pca', PCA(n_components=pca, svd_solver='full', random_state=SEED)))
    steps.append(('svc', SVC(C=C, gamma=gamma, kernel='rbf', class_weight='balanced', random_state=SEED)))
    return Pipeline(steps)

C_VALUES = [1.0, 2.0, 3.0, 4.0, 5.0, 7.0]
GAMMA_VALUES = [0.01, 0.015, 0.0175, 0.02, 0.025, 0.03, 0.04, 0.05]
PCA_VALUES = [None, 0.90, 0.95, 0.99]
rows = []
start = time.time()
total = len(REP_KINDS) * len(PCA_VALUES) * len(C_VALUES) * len(GAMMA_VALUES)
done = 0
print('Total configurations:', total)

for rep in REP_KINDS:
    X = build_rep(V, rep)
    for pca in PCA_VALUES:
        for C in C_VALUES:
            for gamma in GAMMA_VALUES:
                fold_scores = []
                for tr, ho in gkf.split(X, groups=val.engine_id):
                    model = make_svc(C, gamma, pca)
                    y_tr = val.label.iloc[tr].astype(str)
                    y_ho = val.label.iloc[ho].astype(str)
                    model.fit(X[tr], y_tr)
                    pred = model.predict(X[ho])
                    fold_scores.append(macro_f1(y_ho, pred))
                rows.append({
                    'representation': rep,
                    'pca': pca,
                    'C': C,
                    'gamma': gamma,
                    'raw_score': float(np.mean(fold_scores)),
                    'std': float(np.std(fold_scores)),
                    'min_fold': float(np.min(fold_scores))
                })
                done += 1
                if done == 1 or done % 100 == 0 or done == total:
                    print(f'{done}/{total}  elapsed={(time.time()-start)/60:.1f} min')

results = pd.DataFrame(rows).sort_values(['raw_score', 'min_fold'], ascending=False).reset_index(drop=True)
results.to_csv('svc_representation_search.csv', index=False)
print(f'Finished in {(time.time()-start)/60:.1f} min')
display(results.head(25))

In [ ]:
print('TOP BY MEAN SCORE')
display(results.head(20))
print('TOP BY WORST-FOLD SCORE')
display(results.sort_values(['min_fold', 'raw_score'], ascending=False).head(20))
best = results.iloc[0]
print('BEST CONFIGURATION')
print(best.to_string())

## OOF decision-margin diagnostic

This section evaluates confidence margins using out-of-fold predictions from the best configuration. It does not read `test.csv` and does not train on test data.

In [ ]:
rep = str(best['representation'])
pca = None if pd.isna(best['pca']) else float(best['pca'])
C = float(best['C'])
gamma = float(best['gamma'])
X = build_rep(V, rep)
classes = np.array(sorted(val.label.astype(str).unique()))
oof_dec = np.full((len(val), len(classes)), -np.inf, dtype=float)
oof_pred = np.empty(len(val), dtype=object)

for tr, ho in gkf.split(X, groups=val.engine_id):
    model = make_svc(C, gamma, pca)
    model.fit(X[tr], val.label.iloc[tr].astype(str))
    decision = model.decision_function(X[ho])
    model_classes = np.asarray(model.named_steps['svc'].classes_)
    if decision.ndim == 1 and len(model_classes) == 2:
        decision = np.column_stack([-decision, decision])
    for j, cls in enumerate(model_classes):
        idx = np.where(classes == cls)[0][0]
        oof_dec[ho, idx] = decision[:, j]
    oof_pred[ho] = classes[np.argmax(oof_dec[ho], axis=1)]

baseline = macro_f1(val.label.astype(str), oof_pred)
print(f'OOF baseline Macro-F1: {baseline:.6f}')
margin = np.sort(oof_dec, axis=1)[:, -1] - np.sort(oof_dec, axis=1)[:, -2]
for q in [0.00, 0.02, 0.05, 0.10, 0.15, 0.20]:
    cutoff = np.quantile(margin, q)
    pred = oof_pred.copy()
    if 'unknown' in classes:
        pred[margin < cutoff] = 'unknown'
    print(f'abstain q={q:.2f} cutoff={cutoff:.4f} Macro-F1={macro_f1(val.label.astype(str), pred):.6f} abstained={(margin < cutoff).mean():.1%}')

## Important

This research notebook intentionally stops before pseudo-labeling, severity training and test inference. The original SVC notebook is unchanged.